### Loading Test Data and Vocab

In [9]:
import torch
import pandas as pd
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

from utils import load_config, load_vocab
from encoder import Encoder
from decoder import Decoder
from seq2Seq import Seq2Seq

In [10]:
config = load_config()
vocab = load_vocab()
vocab_reversed = load_vocab(rev=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [11]:
from utils import load_sets, get_dataloaders

train_set, val_set, test_set = load_sets()

train_loader, val_loader, test_loader = get_dataloaders(
    train_set,
    val_set,
    test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

In [12]:
model_a = torch.load(
    "models/model_a/model_a_baseline.pt",
    map_location=device,
    weights_only=False
)

model_a.to(device)
model_a.eval()



Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(13617, 256, padding_idx=0)
    (gru): GRU(256, 500, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(13617, 256, padding_idx=0)
    (gru): GRU(256, 500, num_layers=2, batch_first=True, dropout=0.4)
    (linear): Linear(in_features=500, out_features=13617, bias=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
)

In [13]:


model_b = torch.load(
    "models/model_b/model_b_baseline.pt",
    map_location=device,
    weights_only=False
)

model_b.to(device)
model_b.eval()

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(13617, 256, padding_idx=0)
    (gru): GRU(256, 500, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(13617, 256, padding_idx=0)
    (gru): GRU(256, 500, num_layers=2, batch_first=True, dropout=0.4)
    (attn): Linear(in_features=500, out_features=500, bias=True)
    (context_linear): Linear(in_features=1000, out_features=500, bias=True)
    (linear): Linear(in_features=500, out_features=13617, bias=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
)

In [14]:
from inference import generate_response

### BLEU Evaluation

In [16]:
from bleu import compute_bleu

In [22]:
bleu_a = compute_bleu(
    model=model_a,
    loader=test_loader,
    vocab_reversed=vocab_reversed,
    config=config,
    device=device,
    n_batches=10
)

print("Model A BLEU:", bleu_a)

Computing BLEU for epoch...
  BLEU: batch 1/10...
  BLEU: batch 4/10...
  BLEU: batch 7/10...
  BLEU: batch 10/10...
Model A BLEU: 0.001449452382397321


In [23]:
bleu_b = compute_bleu(
    model=model_b,
    loader=test_loader,
    vocab_reversed=vocab_reversed,
    config=config,
    device=device,
    n_batches=10
)

print("Model B BLEU:", bleu_b)

Computing BLEU for epoch...
  BLEU: batch 1/10...
  BLEU: batch 4/10...
  BLEU: batch 7/10...
  BLEU: batch 10/10...
Model B BLEU: 0.0010356605072525625


### Run Evaluation

### Comparison Table

In [24]:
summary_df = pd.DataFrame({
    "model": ["Model A", "Model B"],
    "bleu": [bleu_a, bleu_b],
    "batches_evaluated": [10, 10]
})

summary_df

,model,bleu,batches_evaluated
0,Model A,0.001449,10
1,Model B,0.001036,10


In [25]:
def clean_text(text):
    return text.replace("<S0>", "").replace("<S1>", "").replace("<SEP>", "").strip()

### Showcasing Responses

In [32]:
prompts = [
    "how do i update ubuntu",
    "how do i remove a directory",
    "how do i extract a zip file",
    "how do i find my ip address",
    "how do i check ubuntu version",
    "what is apt",
    "what does sudo do",
    "how can i update my ubuntu system",
    "delete a folder in linux",
    "i am new to linux and want to update ubuntu safely"
]

qual_rows = []

for p in prompts:
    text_for_model = p if p.startswith("<S0>") else f"<S0> {p}"

    out_a = generate_response(
        model=model_a,
        text=text_for_model,
        vocab=vocab,
        vocab_reversed=vocab_reversed,
        config=config,
        device=device,
        top_k=1,
        repetition_penalty=1.0
    )

    out_b = generate_response(
        model=model_b,
        text=text_for_model,
        vocab=vocab,
        vocab_reversed=vocab_reversed,
        config=config,
        device=device,
        top_k=1,
        repetition_penalty=1.0
    )

    qual_rows.append({
        "input": p,
        "model_a_prediction": clean_text(out_a),
        "model_b_prediction": clean_text(out_b)
    })

qual_df = pd.DataFrame(qual_rows)
qual_df

,input,model_a_prediction,model_b_prediction
0,how do i update ubuntu,sudo you <UNK>,sudo apt-get update && apt-get upgrade
1,how do i remove a directory,why you want to,rm -rf dir
2,how do i extract a zip file,you can <UNK>,tar xvfz filename.tar.gz
3,how do i find my ip address,you you to,ifconfig ip ip
4,how do i check ubuntu version,you can the,"check synaptic, <UNK>"
5,what is apt,i think is,gcc <UNK> <UNK>
6,what does sudo do,i <UNK> it,its <UNK> a
7,how can i update my ubuntu system,you you to,sudo apt-get update && apt-get upgrade
8,delete a folder in linux,i <UNK> the,i <UNK> folder
9,i am new to linux and want to update ubuntu sa...,i you to,i you to


### Side by Side BLEU

In [28]:
sample_test = test_set.sample(50, random_state=42)

In [31]:
smooth = SmoothingFunction().method1

rows = []

for _, row in sample_test.iterrows():
    input_clean = clean_text(row["input"])
    ref_clean = clean_text(row["response"])

    text_for_model = row["input"] if row["input"].startswith("<S0>") else f"<S0> {input_clean}"

    pred_a = generate_response(
        model=model_a,
        text=text_for_model,
        vocab=vocab,
        vocab_reversed=vocab_reversed,
        config=config,
        device=device,
        top_k=1,
        repetition_penalty=1.0
    )
    pred_a_clean = clean_text(pred_a)

    pred_b = generate_response(
        model=model_b,
        text=text_for_model,
        vocab=vocab,
        vocab_reversed=vocab_reversed,
        config=config,
        device=device,
        top_k=1,
        repetition_penalty=1.0
    )
    pred_b_clean = clean_text(pred_b)

    bleu_a_sent = sentence_bleu(
        [ref_clean.split()],
        pred_a_clean.split(),
        smoothing_function=smooth
    )

    bleu_b_sent = sentence_bleu(
        [ref_clean.split()],
        pred_b_clean.split(),
        smoothing_function=smooth
    )

    rows.append({
        "input": input_clean,
        "reference": ref_clean,
        "model_a_prediction": pred_a_clean,
        "model_a_sentence_bleu": bleu_a_sent,
        "model_b_prediction": pred_b_clean,
        "model_b_sentence_bleu": bleu_b_sent
    })

comparison_df = pd.DataFrame(rows)
comparison_df.head(10)

,input,reference,model_a_prediction,model_a_sentence_bleu,model_b_prediction,model_b_sentence_bleu
0,hey anyone knows anything about exim?,only that debian etch's aptitude can't remove ...,i know but is a,0.000000,ask ask question,0.000000
1,what's a good app to make mp3s from a cd (with...,: sound-juicer make good oggs,i think is a of,0.000000,k3b is brasero,0.000000
2,why card exactly?,geforce4 mx on board graphics,i <UNK> to,0.000000,because i integrated <UNK>,0.000000
3,"hi, how to change number of desktops with desk...",have you installed compizconfig-settings-manag...,<UNK> you use,0.000766,go click <UNK>,0.000766
4,ok but i am using lucid. is that package pres...,am i missing something?? how about a webbrowse...,i think can the,0.010873,no is <UNK>,0.000000
5,is the system you want to virtualize windows?,yes but it is an exsisting system,i think was,0.000000,i want to <UNK>,0.000000
6,how do i redirect output from a program that i...,i think it's 2>&1,you <UNK> you,0.000000,kill <UNK> programname,0.000000
7,hi all! need help: how to change single click ...,edit/preferences/behaviour in nautilus,you <UNK> the,0.000000,you the <UNK>,0.000000
8,hello everyone how can i update from 9.10 bet...,sudo apt-get update; sudo apt-get dist-upgrade,you to the,0.000000,you the update,0.000000
9,"hey guys, how can i kill arts? anyone here ha...","yea, the butler did it",you you to,0.000000,i a <UNK>,0.000000
